# Run one experiment (any order, method and seed)
One run = three stages in a row, with one replay method. Set the three lines in **Settings** and Run All.

* **Needs a GPU:** Runtime → Change runtime type → **L4**. About 1 to 1.5 hours per run.
* Keep your Mac awake: `caffeinate -dims` in Terminal. Only one notebook on the GPU at a time.
* **If Colab disconnects:** set `START_STAGE` to the stage that did not finish and Run All again.
* Everything is saved to the team Drive after every stage: adapter, loss history, replay log,
  per-pair margins, the memory buffer and the results table.

In [ ]:
import os, sys

if os.path.exists("/content"):   # on Colab: get the latest code + data from GitHub
    if not os.path.exists("/content/mfr-dpo"):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    !git -C /content/mfr-dpo pull -q
    !pip install -q peft bitsandbytes
    REPO = "/content/mfr-dpo"
else:
    REPO = ".."

sys.path.insert(0, f"{REPO}/src")
import importlib, mfr_data, mfr_dpo, mfr_utils, mfr_replay
for module in (mfr_data, mfr_dpo, mfr_utils, mfr_replay):
    importlib.reload(module)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a GPU")

In [ ]:
import getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF token (read-only), or press Enter to skip: ")

## Settings
Change `ORDER_ID`, `METHOD` and `SEED`. Everything else is frozen for all runs (see the handbook).

In [ ]:
ORDER_ID = 2                 # 1: helpful -> safe -> quality      2: safe -> helpful -> quality
METHOD = "mfr"               # none | random | lowest_margin | mfr
SEED = 0                     # 0 or 1

# frozen settings, do not change
ORDERS = {1: ["helpful", "safe", "quality"], 2: ["safe", "helpful", "quality"]}
ORDER = ORDERS[ORDER_ID]
LR, BETA = 1e-4, 0.1
NEW_PER_STEP, OLD_PER_STEP, REFRESHES = 16, 2, 5      # 2 old pairs out of 18 per step
BUFFER_SIZE = 500
MAX_SHARE_PER_DATASET = 0.75   # in stage 3, one earlier dataset may take at most 75% of the replay slots
EVAL_SETS = ["safe", "helpful", "quality"]

RUN_NAME = f"o{ORDER_ID}_{METHOD}_s{SEED}"
START_STAGE = 1              # after a disconnect: the first stage that did NOT finish

# Your path to the team folder in Drive (the only line that differs between people)
DRIVE_DIR = "/content/drive/MyDrive/CSCI544/mfr-dpo"
RUN_DIR = f"{DRIVE_DIR}/runs/{RUN_NAME}"

# Stage 1 is the same for every method, so we train it once per order+seed and reuse it.
# For order 2 + seed 0 the pilot already trained it. Set to None to train stage 1 in this run.
STAGE1_FROM = f"{DRIVE_DIR}/runs/pilot_safe_first/stage1_safe" if (ORDER_ID == 2 and SEED == 0) else None

print(RUN_NAME, "|", " -> ".join(ORDER), "| replay:", METHOD)

In [ ]:
import json
from google.colab import drive
drive.mount("/content/drive")    

if START_STAGE == 1 and os.path.exists(f"{RUN_DIR}/results.csv"):
    raise RuntimeError(f"{RUN_NAME} already has results. Use a new name, or set START_STAGE to resume.")
os.makedirs(RUN_DIR, exist_ok=True)

settings = {"run_name": RUN_NAME, "order_id": ORDER_ID, "order": ORDER, "method": METHOD, "seed": SEED,
            "lr": LR, "beta": BETA, "new_per_step": NEW_PER_STEP, "old_per_step": OLD_PER_STEP,
            "refreshes": REFRESHES, "buffer_size": BUFFER_SIZE,
            "max_share_per_dataset": MAX_SHARE_PER_DATASET, "stage1_from": STAGE1_FROM}
with open(f"{RUN_DIR}/settings.json", "w") as f:
    json.dump({**settings, **mfr_utils.run_info(REPO)}, f, indent=2)

splits = mfr_data.load_splits(f"{REPO}/data")
print("saving to:", RUN_DIR)

## Train
For each stage: train (with replay from stage 2 on), save the adapter, score the val pairs of all three
datasets, then store 500 of this stage's training pairs in the memory buffer with the margins they have
right now (their "peak"), and re-measure the older pairs already in the buffer.

In [ ]:
import pandas as pd
from mfr_replay import ReplayBuffer

def stage_dir(stage):
    return f"{RUN_DIR}/stage{stage}_{ORDER[stage - 1]}"

def score_val_sets(model, tokenizer, stage, trained_on):
    rows = []
    for name in EVAL_SETS:
        scores = mfr_dpo.score_pairs(model, tokenizer, splits[name]["val"], beta=BETA,
                                     desc=f"scoring {name} val")
        scores.to_csv(f"{stage_dir(stage)}/margins_{name}_val.csv")
        rows.append({"run_name": RUN_NAME, "order_id": ORDER_ID, "method": METHOD, "seed": SEED,
                     "stage": stage, "trained_on": trained_on, "eval_set": name,
                     **mfr_dpo.summarize(scores)})
    return pd.DataFrame(rows)

def update_buffer(model, tokenizer, buffer, stage):
    """Add this stage's pairs (with their peak margins) and re-measure the ones already stored."""
    name = ORDER[stage - 1]
    candidates = buffer.candidates(name, splits[name]["train"])
    peak = mfr_dpo.score_pairs(model, tokenizer, candidates, beta=BETA,
                               desc=f"buffer: peak margins for {name}")["margin"]
    buffer.add_stage(name, candidates, peak)
    older = buffer.rows(exclude_dataset=name)
    if len(older):
        current = mfr_dpo.score_pairs(model, tokenizer, older, beta=BETA,
                                      desc="buffer: re-measuring older pairs")["margin"]
        buffer.set_current(current)
    buffer.to_csv(f"{stage_dir(stage)}/buffer.csv")
    print(f"buffer: {len(buffer)} pairs " + str(buffer.rows()['dataset'].value_counts().to_dict()))
    return buffer

In [ ]:
results = pd.DataFrame()
buffer = ReplayBuffer(size=BUFFER_SIZE, seed=SEED)
first_stage = START_STAGE

# where do we start from?
if START_STAGE > 1:                                   # resuming this run after a disconnect
    previous = stage_dir(START_STAGE - 1)
    buffer = ReplayBuffer.from_csv(f"{previous}/buffer.csv", size=BUFFER_SIZE, seed=SEED)
    results = pd.read_csv(f"{RUN_DIR}/results.csv")
    results = results[results["stage"] < START_STAGE]
elif STAGE1_FROM:                                     # reuse a stage 1 trained earlier
    previous = STAGE1_FROM
    first_stage = 2
else:
    previous = None

mfr_utils.seed_everything(SEED)                       # before load_model: same starting adapter every time
model, tokenizer = mfr_dpo.load_model(adapter_path=previous)
print("starting from:", previous or "a fresh adapter")

if STAGE1_FROM and START_STAGE == 1:                  # rebuild stage 1's bookkeeping without retraining
    os.makedirs(stage_dir(1), exist_ok=True)
    source_results = f"{os.path.dirname(STAGE1_FROM)}/results.csv"
    if os.path.exists(source_results):
        stage1 = pd.read_csv(source_results)
        stage1 = stage1[stage1["stage"] == 1].assign(run_name=RUN_NAME, method=METHOD,
                                                     order_id=ORDER_ID, seed=SEED)
        results = pd.concat([results, stage1], ignore_index=True)
    buffer = update_buffer(model, tokenizer, buffer, 1)

In [ ]:
for stage in range(first_stage, len(ORDER) + 1):
    name = ORDER[stage - 1]
    print(f"\n===== Stage {stage}/{len(ORDER)}: training on {name} (replay: {METHOD}) =====")
    os.makedirs(stage_dir(stage), exist_ok=True)
    mfr_utils.seed_everything(mfr_utils.stage_seed(SEED, stage))

    history, replay_log = mfr_dpo.train_stage_replay(
        model, tokenizer, splits[name]["train"], buffer=buffer, method=METHOD,
        beta=BETA, lr=LR, new_per_step=NEW_PER_STEP, old_per_step=OLD_PER_STEP,
        refreshes=REFRESHES, max_share_per_dataset=MAX_SHARE_PER_DATASET,
        seed=mfr_utils.stage_seed(SEED, stage), desc=f"stage {stage}/{len(ORDER)}: {name}")

    model.save_pretrained(stage_dir(stage))
    history.to_csv(f"{stage_dir(stage)}/history.csv", index=False)
    replay_log.to_csv(f"{stage_dir(stage)}/replay_log.csv", index=False)

    stage_results = score_val_sets(model, tokenizer, stage, name)
    results = pd.concat([results, stage_results], ignore_index=True)
    results.to_csv(f"{RUN_DIR}/results.csv", index=False)

    buffer = update_buffer(model, tokenizer, buffer, stage)
    print("saved stage", stage, "to", stage_dir(stage))
    display(stage_results.set_index("eval_set")[["accuracy", "accuracy_sum", "mean_margin"]])

## This run's results
These cells only read the saved files, so you can run them later without a GPU.

In [ ]:
results = pd.read_csv(f"{RUN_DIR}/results.csv")
table = results.pivot(index="eval_set", columns="stage", values="accuracy").loc[EVAL_SETS]
table.columns = [f"after stage {s} ({ORDER[s - 1]})" for s in table.columns]
table

In [ ]:
# how much of each earlier stage survived to the end
last = len(ORDER)
rows = []
for k, name in enumerate(ORDER[:-1], start=1):
    after = results[(results["stage"] == k) & (results["eval_set"] == name)].iloc[0]
    end = results[(results["stage"] == last) & (results["eval_set"] == name)].iloc[0]
    rows.append({"dataset": name, "learned at stage": k,
                 "accuracy right after (%)": after["accuracy"], "accuracy at the end (%)": end["accuracy"],
                 "change (points)": round(end["accuracy"] - after["accuracy"], 1),
                 "mean margin kept (%)": round(100 * end["mean_margin"] / after["mean_margin"])
                                         if after["mean_margin"] > 0 else None})
pd.DataFrame(rows).set_index("dataset").astype(object).T

## Compare the methods (once more runs of this order and seed exist)

In [ ]:
import glob

comparison = []
for path in sorted(glob.glob(f"{DRIVE_DIR}/runs/o{ORDER_ID}_*_s{SEED}/results.csv")):
    other = pd.read_csv(path)
    last = other["stage"].max()
    row = {"run": os.path.basename(os.path.dirname(path))}
    for name in ORDER[:-1]:                                   # the stages that can be forgotten
        learned_at = ORDER.index(name) + 1
        after = other[(other["stage"] == learned_at) & (other["eval_set"] == name)]["accuracy"]
        end = other[(other["stage"] == last) & (other["eval_set"] == name)]["accuracy"]
        if len(after) and len(end):
            row[f"{name}: after"] = after.iloc[0]
            row[f"{name}: end"] = end.iloc[0]
            row[f"{name}: change"] = round(end.iloc[0] - after.iloc[0], 1)
    newest = ORDER[-1]
    end_new = other[(other["stage"] == last) & (other["eval_set"] == newest)]["accuracy"]
    if len(end_new):
        row[f"{newest} (new, higher is better)"] = end_new.iloc[0]
    comparison.append(row)

pd.DataFrame(comparison).set_index("run") if comparison else "no runs found yet"

**How to read the comparison:** the replay methods should lose **less** on the earlier datasets
(the "change" columns, less negative is better) than `none`, without losing much on the last
dataset. MFR wins if it keeps more than `random` at the same cost.